In [82]:
import numpy as np

In [93]:
x = np.array([1, 2, 3, 4]).reshape(-1, 1)
y = np.array([1, 4, 9, 16]).reshape(-1, 1)

X_train = x
y_train = y
N = X_train.shape[1]

def weights(input_size, output_size, seed = None):
    if seed is not None:
        np.random.seed(seed)
    limit = np.sqrt(6 / (input_size + output_size))
    weights = np.random.uniform(-limit, limit, size = (input_size, output_size))
    return weights

def bias(size):
    return np.zeros((1,size))

def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(float)

def mse(y_pred, y_true):
    return np.mean((y_pred - y_true) ** 2)

In [94]:
W1 = weights(X_train.shape[1], 16, seed = 42)
W2 = weights(16, 8, seed = 42)
W3 = weights(8, 1, seed = 42)
B1 = bias(16)
B2 = bias(8)
B3 = bias(1)

In [95]:
def forward(X_train):
    Z1 = np.dot(X_train, W1) + B1
    A1 = relu(Z1)
    Z2 = np.dot(A1, W2) + B2
    A2 = relu(Z2)
    Z3 = np.dot(A2, W3) + B3
    return Z1, A1, Z2, A2, Z3

In [96]:
def backward(X, y, Z1, A1, Z2, A2, Z3, lr=0.01):
    global W1, W2, W3, B1, B2, B3

    # dL/dZ3
    dZ3 = (2 / N) * (Z3 - y)

    dW3 = np.dot(A2.T, dZ3)
    dB3 = np.sum(dZ3, axis=0, keepdims=True)
    
    dA2 = np.dot(dZ3, W3.T)
    dZ2 = dA2 * relu_derivative(Z2)

    dW2 = np.dot(A1.T, dZ2)
    dB2 = np.sum(dZ2, axis=0, keepdims=True)

    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * relu_derivative(Z1)

    dW1 = np.dot(X.T, dZ1)
    dB1 = np.sum(dZ1, axis=0, keepdims=True)

    # update
    W3 -= lr * dW3
    B3 -= lr * dB3
    W2 -= lr * dW2
    B2 -= lr * dB2
    W1 -= lr * dW1
    B1 -= lr * dB1

In [ ]:
epochs = 5
lr = 0.01

for epoch in range(epochs):
    Z1, A1, Z2, A2, Z3 = forward(X_train)
    loss = mse(Z3, y_train)
    backward(X_train, y_train, Z1, A1, Z2, A2, Z3, lr)
    print(f"Epoch {epoch}, Loss: {loss:.4f}")